# Regression with PyTorch V2.1 (oracle) — Income, Neural Additive Model + robust head, k=5

**Version:** V2.1 (k-fold validation — novel architectures). **Role:** reference oracle for the V2.1
scratch twin. Same `NAM` architecture and hypers as `regression_pytorch_V2_0.ipynb` (13 per-feature
subnets, `log1p(Income)` z-scored target, Huber(δ=0.5) loss, Adam+cosine+wd(1e-4), 100 epochs,
best-val-RMSE checkpoint), copied verbatim and wrapped in a 5-fold `KFold` loop over the full
2,212-row dropped-target set instead of one fixed 80/20 split. Weights → `./weights_V2_1/`.

In [1]:
import json, os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
os.makedirs("weights_V2_1", exist_ok=True)
DELTA = 0.5

## 1. LOAD DATA (identical to V2.0)

In [2]:
df = pd.read_csv("marketing_campaign.csv", sep="\t")
df = df.drop(columns=["ID", "Z_CostContact", "Z_Revenue"])
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%d-%m-%Y")
ref_date = df["Dt_Customer"].max()
df["Customer_Tenure_Days"] = (ref_date - df["Dt_Customer"]).dt.days
df["Age"] = 2014 - df["Year_Birth"]
df = df[~df["Year_Birth"].isin([1893, 1899, 1900])].copy()
df = df[df["Income"] != 666666].copy()  # drop $666,666 income outlier (same treatment as Year_Birth sentinels)
df["Marital_Status"] = df["Marital_Status"].replace({"Absurd": "Single", "YOLO": "Single", "Alone": "Single"})
df = df.dropna(subset=["Income"]).copy()
print("cleaned:", df.shape, "| Income max:", f"{df['Income'].max():,.0f}", "| median:", f"{df['Income'].median():,.0f}")
REG_FEATURES = ["NumCatalogPurchases","MntMeatProducts","MntWines","NumWebVisitsMonth","MntSweetProducts",
    "MntFruits","NumWebPurchases","MntGoldProds","NumStorePurchases","MntFishProducts","Kidhome","Age","NumDealsPurchases"]
X_full = df[REG_FEATURES].values; y_raw_full = df["Income"].values.astype(float)
print(f"features: {len(REG_FEATURES)} dims (all numeric)")

class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(np.asarray(X, dtype=np.float32))
        self.y = torch.from_numpy(np.asarray(y, dtype=np.float32)).view(-1, 1)
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i): return self.X[i], self.y[i]

cleaned: (2212, 28) | Income max: 162,397 | median: 51,371
features: 13 dims (all numeric)


## 2. MODEL: Neural Additive Model (identical to V2.0)

In [3]:
class SubNet(nn.Module):
    def __init__(self, h=64):
        super().__init__()
        self.fc1 = nn.Linear(1, h); self.fc2 = nn.Linear(h, 1); self.relu = nn.ReLU()
        nn.init.kaiming_normal_(self.fc1.weight, nonlinearity="relu"); nn.init.zeros_(self.fc1.bias)
        nn.init.kaiming_normal_(self.fc2.weight, nonlinearity="relu"); nn.init.zeros_(self.fc2.bias)
    def forward(self, xj): return self.fc2(self.relu(self.fc1(xj)))

class NAM(nn.Module):
    def __init__(self, n_feat=13, h=64):
        super().__init__()
        self.subs = nn.ModuleList([SubNet(h) for _ in range(n_feat)])
        self.b0 = nn.Parameter(torch.zeros(1))
    def forward(self, x):
        return sum(sub(x[:, j:j+1]) for j, sub in enumerate(self.subs)) + self.b0

device = torch.device("cpu")

## K-fold training (k=5) — fresh NAM per fold, best-val-RMSE checkpoint on own fold

In [4]:
K = 5
kf = KFold(n_splits=K, shuffle=True, random_state=SEED)
fold_rows = []
for fold, (tr_idx, va_idx) in enumerate(kf.split(X_full), start=1):
    Xtr_f, Xva_f = X_full[tr_idx], X_full[va_idx]
    ytr_raw, yva_raw = y_raw_full[tr_idx], y_raw_full[va_idx]
    sc_f = StandardScaler().fit(Xtr_f)
    Xtr_fs, Xva_fs = sc_f.transform(Xtr_f), sc_f.transform(Xva_f)
    ytr_log = np.log1p(ytr_raw).reshape(-1, 1)
    ysc_f = StandardScaler().fit(ytr_log)
    ytr_z = ysc_f.transform(ytr_log).ravel()
    train_ds_f = TabularDataset(Xtr_fs, ytr_z)
    train_loader_f = DataLoader(train_ds_f, batch_size=64, shuffle=True,
                                 generator=torch.Generator().manual_seed(SEED + fold))

    def to_dollar(z, ysc=ysc_f):
        return np.expm1(ysc.inverse_transform(np.asarray(z).reshape(-1, 1)).ravel())

    torch.manual_seed(SEED + fold)
    m = NAM(n_feat=len(REG_FEATURES), h=64).to(device)
    crit_f = nn.HuberLoss(delta=DELTA)
    opt_f = torch.optim.Adam(m.parameters(), lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=1e-4)
    sch_f = torch.optim.lr_scheduler.CosineAnnealingLR(opt_f, T_max=100)

    Xv = torch.from_numpy(Xva_fs.astype(np.float32)).to(device)
    best_rmse, best_state = float("inf"), None
    for epoch in range(1, 101):
        m.train()
        for xb, yb in train_loader_f:
            xb, yb = xb.to(device), yb.to(device)
            opt_f.zero_grad()
            crit_f(m(xb), yb).backward()
            opt_f.step()
        sch_f.step()
        m.eval()
        with torch.no_grad():
            pred_z = m(Xv).cpu().numpy().ravel()
        pred_v = to_dollar(pred_z)
        rmse_v = float(np.sqrt(mean_squared_error(yva_raw, pred_v)))
        if rmse_v < best_rmse:
            best_rmse = rmse_v
            best_state = {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}

    m.load_state_dict(best_state)
    m.eval()
    with torch.no_grad():
        pred_v = to_dollar(m(Xv).cpu().numpy().ravel())
    mse_v = float(mean_squared_error(yva_raw, pred_v))
    fold_rows.append({
        "fold": fold,
        "test_MSE": round(mse_v, 1),
        "rmse": round(float(np.sqrt(mse_v)), 1),
        "mae": round(float(mean_absolute_error(yva_raw, pred_v)), 1),
        "r2": round(float(r2_score(yva_raw, pred_v)), 4),
    })
    print(f"fold {fold}: RMSE={fold_rows[-1]['rmse']:.1f} R2={fold_rows[-1]['r2']:.4f}")

kfold_df = pd.DataFrame(fold_rows)
summary = {k: {"mean": round(float(kfold_df[k].mean()), 4), "std": round(float(kfold_df[k].std()), 4)}
           for k in ("test_MSE", "rmse", "mae", "r2")}
print(kfold_df.to_string(index=False))
print(json.dumps(summary, indent=2))
with open("weights_V2_1/regression_pytorch_kfold_metrics.json", "w") as f:
    json.dump({"k": K, "folds": fold_rows, "summary": summary}, f, indent=2)

fold 1: RMSE=11387.5 R2=0.7270


fold 2: RMSE=9712.0 R2=0.7889


fold 3: RMSE=10213.2 R2=0.7674


fold 4: RMSE=11845.3 R2=0.7153


fold 5: RMSE=11628.5 R2=0.6988
 fold    test_MSE    rmse    mae     r2
    1 129674769.5 11387.5 7347.8 0.7270
    2  94322335.6  9712.0 7178.9 0.7889
    3 104310366.5 10213.2 6933.4 0.7674
    4 140310798.5 11845.3 8141.0 0.7153
    5 135222165.2 11628.5 7388.9 0.6988
{
  "test_MSE": {
    "mean": 120768087.06,
    "std": 20250933.7668
  },
  "rmse": {
    "mean": 10957.3,
    "std": 939.2261
  },
  "mae": {
    "mean": 7398.0,
    "std": 452.3577
  },
  "r2": {
    "mean": 0.7395,
    "std": 0.0375
  }
}
